## Build FAISS HNSW Index

*(Assumes `EMB_DIR`, `MODEL_NAME`, `CFG_NAME`, `PROJECT_ROOT` from `00_config.ipynb`.)*

In [ ]:
# --- HNSW index builder: FAISS IndexHNSWFlat (cosine/IP) ---
import os, json, time
from pathlib import Path

import numpy as np
import psutil

try:
    import faiss  # CPU FAISS
except ImportError as e:
    raise RuntimeError("faiss is required. Install with: pip install faiss-cpu") from e

# ------------------ CONFIG ------------------

# Use the same EMB_DIR from your embedding step
EMB_PATH   = Path(EMB_DIR / "emb.npy")
IDS_PATH   = Path(EMB_DIR / "ids.npy")
META_PATH  = Path(EMB_DIR / "meta.json")

# HNSW hyperparameters
HNSW_M             = int(os.environ.get("HNSW_M", 32))
HNSW_EF_CONSTRUCT  = int(os.environ.get("HNSW_EF_CONSTRUCT", 200))
HNSW_EF_SEARCH     = int(os.environ.get("HNSW_EF_SEARCH", 64))

# Add in batches
ADD_BATCH_SIZE     = int(os.environ.get("HNSW_ADD_BATCH_SIZE", 100_000))

# Save index in a consistent directory (avoid mixing runs)
MODEL_DIRNAME = MODEL_NAME.replace("/", "_")
INDEX_DIR = PROJECT_ROOT / "indexes" / MODEL_DIRNAME / CFG_NAME / f"hnsw_M{HNSW_M}_efC{HNSW_EF_CONSTRUCT}_efS{HNSW_EF_SEARCH}"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = INDEX_DIR / "index.faiss"
INDEX_META = INDEX_DIR / "meta.json"

# Limit FAISS threads (optional)
try:
    n_threads = max(1, min(os.cpu_count() or 1, 8))
    faiss.omp_set_num_threads(n_threads)
except Exception:
    n_threads = None

# ------------------ LOAD EMBEDDINGS (MMAP) ------------------
if not EMB_PATH.exists():
    raise FileNotFoundError(f"Embeddings file not found: {EMB_PATH}")
if not IDS_PATH.exists():
    raise FileNotFoundError(f"IDs file not found: {IDS_PATH}")

# Read meta if present
if META_PATH.exists():
    with open(META_PATH, "r", encoding="utf-8") as f:
        emb_meta = json.load(f)
    D = int(emb_meta.get("dim"))
    N = int(emb_meta.get("n_vecs"))
else:
    X_tmp = np.load(EMB_PATH, mmap_mode="r")
    if X_tmp.ndim != 2:
        raise ValueError(f"Expected (N,D) embeddings array; got shape {X_tmp.shape}")
    N, D = int(X_tmp.shape[0]), int(X_tmp.shape[1])

# Memory-map embeddings and ids
X = np.load(EMB_PATH, mmap_mode="r")
ids = np.load(IDS_PATH, mmap_mode="r")

if len(ids) != N:
    raise ValueError(f"IDs length {len(ids)} != embeddings N {N}")

need_cast = (X.dtype != np.float32)

# ------------------ BUILD INDEX (IP/COSINE) ------------------
# Normalized embeddings => cosine == inner product
try:
    base = faiss.IndexHNSWFlat(D, HNSW_M, faiss.METRIC_INNER_PRODUCT)
except TypeError:
    base = faiss.IndexHNSWFlat(D, HNSW_M)
    if hasattr(base, "metric_type"):
        base.metric_type = faiss.METRIC_INNER_PRODUCT

base.hnsw.efConstruction = HNSW_EF_CONSTRUCT
base.hnsw.efSearch = HNSW_EF_SEARCH

# Wrap to store your faiss_id values
index = faiss.IndexIDMap2(base)

# Remove stale artifact before writing
try:
    if INDEX_PATH.exists():
        INDEX_PATH.unlink()
except PermissionError:
    raise RuntimeError(f"{INDEX_PATH} is in use; close readers and retry.")

proc = psutil.Process(os.getpid())
def rss_mb() -> float:
    return proc.memory_info().rss / (1024 * 1024)

peak_rss_mb = rss_mb()

t0 = time.time()
added = 0

for start in range(0, N, ADD_BATCH_SIZE):
    end = min(start + ADD_BATCH_SIZE, N)
    sl = slice(start, end)

    # embeddings batch (float32, C-contiguous)
    xb = np.asarray(X[sl], dtype=np.float32, order="C") if need_cast else np.asarray(X[sl], dtype=np.float32, order="C")
    # ids batch (int64, C-contiguous)
    id_batch = np.asarray(ids[sl], dtype=np.int64, order="C")

    index.add_with_ids(xb, id_batch)

    added += (end - start)
    peak_rss_mb = max(peak_rss_mb, rss_mb())

elapsed = time.time() - t0
assert index.ntotal == added, f"Index holds {index.ntotal} but added {added}"

# ------------------ SAVE INDEX + STATS ------------------
faiss.write_index(index, str(INDEX_PATH))
file_size_mb = INDEX_PATH.stat().st_size / (1024 * 1024)

index_meta = {
    "type": "faiss.IndexHNSWFlat + IndexIDMap2",
    "model": MODEL_NAME,
    "embeddings_path": str(EMB_PATH),
    "ids_path": str(IDS_PATH),
    "n_vecs": int(N),
    "dim": int(D),
    "normalized": True,
    "metric": "inner_product",
    "hnsw": {"M": HNSW_M, "efConstruction": HNSW_EF_CONSTRUCT, "efSearch": HNSW_EF_SEARCH},
    "build": {
        "time_sec": round(elapsed, 2),
        "throughput_vecs_per_sec": round(N / elapsed, 1) if elapsed > 0 else None,
        "peak_rss_mb": round(peak_rss_mb, 1),
        "faiss_threads": n_threads,
    },
    "artifacts": {
        "index_path": str(INDEX_PATH),
        "index_disk_size_mb": round(file_size_mb, 1),
        "index_dir": str(INDEX_DIR),
        "meta_path": str(INDEX_META),
    },
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

with open(INDEX_META, "w", encoding="utf-8") as f:
    json.dump(index_meta, f, indent=2)

print(f"[OK] HNSW index saved → {INDEX_PATH}")
print(f"     Vectors: {N} | Dim: {D} | Metric: IP(cosine)")
print(f"     HNSW(M={HNSW_M}, efC={HNSW_EF_CONSTRUCT}, efS={HNSW_EF_SEARCH})")
print(f"     Build time: {elapsed:.2f}s | Peak RSS: {peak_rss_mb:.1f} MB | File: {file_size_mb:.1f} MB")
print(f"     Meta: {INDEX_META}")
